##raw ingestion of source tables with lineage metadata

In [0]:

from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bakehouse_bronze"
SOURCE_SCHEMA = "samples.bakehouse"

TABLES = ["sales_transactions", "sales_customers", "sales_franchises"]

In [0]:
def ingest_to_bronze(table_name: str) -> int:
    """Land a source table into bronze with ingestion metadata. Returns row count"""
    source_path = f"{SOURCE_SCHEMA}.{table_name}"
    target_path = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{table_name}"

    df = (
        spark.table(source_path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_table", F.lit(source_path))
    )
    

    df.write.format("delta").mode("overwrite").saveAsTable(target_path)
    return spark.table(target_path).count()

In [0]:
# Run ingestion for all source tables
row_counts = {}
for table in TABLES:
    row_counts[table] = ingest_to_bronze(table)

##Verification

In [0]:
print(f"{'Table':<25}{'Bronze Table':<35}{'Rows':>10}")
print("-" * 70)
for table, count in row_counts.items():
    bronze_name = f"bronze_{table}"
    print(f"{table:<25}{bronze_name:<35}{count:>10}")

assert all(c > 0 for c in row_counts.values()), "One or more bronze tables ingested 0 rows"
print("\nAll bronze tables ingested successfully.")